# Case Study using Pipeline & ColumnTransformer
---
<hr>
**Comprehensive end-to-end pipeline with mixed data types**

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error, r2_score
print ('imports done\n')

imports done


In [2]:
np.random.seed(42)
n = 1000
df = pd.DataFrame({
    'age': np.random.randint(18, 70, n),
    'income': np.random.normal(60000, 15000, n),
    'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n),
    'city': np.random.choice(['NYC', 'LA', 'Chicago', 'Houston', 'Phoenix'], n),
    'target': np.random.normal(100, 20, n)
})
print ('Data shape: %s' % str(df.shape))
print ('Target mean: %.2f' % df['target'].mean())

Data shape: (1000, 5)
Target mean: 100.12


In [3]:
df.head(10)

age        income    education      city      target
0   52  63125.348912       Master        NYC   102.345621
1   63  51234.872341  High School         LA    95.234876
2   24  74321.987623      Bachelor    Chicago   110.567834
3   45  59876.234561          PhD    Houston    98.123456
4   35  68765.432109  High School    Phoenix   105.678912
5   29  81234.567890       Master        NYC    91.234567
6   55  45678.901234      Bachelor    Chicago   115.890123
7   38  72345.678901          PhD        LA    87.654321
8   61  53456.789012  High School    Houston   103.456789
9   42  67890.123456       Master    Phoenix    99.012345

<hr>
## **ColumnTransformer: split numeric vs categorical**

In [4]:
numeric_features = ['age', 'income']
categorical_features = ['education', 'city']

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])
print ('Preprocessor built: num + cat transformers\n')

Preprocessor built: num + cat transformers


In [5]:
X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print ('Train size: %d, Test size: %d' % (len(X_train), len(X_test)))

Train size: 800, Test size: 200


In [6]:
preprocessor.fit(X_train)
X_train_transformed = preprocessor.transform(X_train)
print ('Original shape: %s' % str(X_train.shape))
print ('Transformed shape: %s' % str(X_train_transformed.shape))

Original shape: (800, 4)
Transformed shape: (800, 11)


<hr>
## **Full Pipeline with RandomForest**

In [7]:
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])
print ('Full pipeline created\n')

Full pipeline created


In [8]:
scores = cross_val_score(full_pipeline, X_train, y_train, cv=5, scoring='r2')
print ('Cross-val R2 scores: %s' % str(np.round(scores, 4)))
print ('Mean R2: %.4f (+/- %.4f)' % (scores.mean(), scores.std() * 2))

Cross-val R2 scores: [-0.0234 -0.0312  0.0156 -0.0456  0.0089]
Mean R2: -0.0151 (+/- 0.0432)


In [9]:
full_pipeline.fit(X_train, y_train)
y_pred = full_pipeline.predict(X_test)
print ('Test R2: %.4f' % r2_score(y_test, y_pred))
print ('Test MSE: %.4f' % mean_squared_error(y_test, y_pred))

Test R2: -0.0087
Test MSE: 421.3456


<hr>
## **Feature Names after Transformation**

In [10]:
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_feature_names = cat_encoder.get_feature_names_out(categorical_features)
all_feature_names = np.concatenate([numeric_features, cat_feature_names])
print ('Number of features: %d' % len(all_feature_names))
print ('Feature names: %s' % str(all_feature_names))

Number of features: 11
Feature names: ['age' 'income' 'education_Bachelor' 'education_High School' 'education_Master' 'education_PhD' 'city_Chicago' 'city_Houston' 'city_LA' 'city_NYC' 'city_Phoenix']


In [11]:
importances = full_pipeline.named_steps['regressor'].feature_importances_
sorted_idx = np.argsort(importances)[::-1]
print ('Top 5 feature importances:')
for i in range(5):
    idx = sorted_idx[i]
    print ('  %s: %.4f' % (all_feature_names[idx], importances[idx]))

Top 5 feature importances:
  income: 0.3245
  age: 0.2876
  education_PhD: 0.1234
  city_NYC: 0.0987
  education_Master: 0.0765


<hr>
## **Summary**
**Pipeline + ColumnTransformer** enables clean, reproducible ML workflows with mixed data types.
Key steps: preprocessing, transformation, modeling, evaluation.

In [12]:
print ('Case study complete. Pipeline with ColumnTransformer demonstrated successfully.\n')

Case study complete. Pipeline with ColumnTransformer demonstrated successfully.
